## Quantifi DSP Demo - Products and Trades

The purpose of this notebook is to demonstrate **Quantifi Data Science Platform (DSP)** capabilities. 

Presented below are some examples of Quantifi framework integration with various popular Data Science and data manipulation and presentation tools available in Python. In particular, we demonstrate the following aspects of the framework: 

- Integration of Quantifi data and pricing models/objects loaded directly from Quantifi Risk into the Jupyter notebook
- Pricing and risk calculations using Quantifi Toolkit directly in the Python code
- Using standard Python libraries (such as *Pandas* and *MatPlotLib*) with Quantifi



***

### Set up Quantifi Toolkit and DSP Access 

**Import packages**

First, we need to load various libraries that we will use in our notebook:

In [1]:
# Import standard packages
import pandas as pd
import numpy as np
import datetime
import time
import os

# Import packages for display formatting/graph plotting
import ipywidgets as widgets
import matplotlib.pyplot as plt

# Import packages Quantifi Toolkit packages 
from Quantifi.Toolkit.Products import Bond
from Quantifi.Toolkit.Pricers import BondPricer
from Quantifi.Toolkit.Base import Dt, DayCount, Frequency, Calendar, QuotingConvention, BumpTarget
from Quantifi.Toolkit.Sensitivity import Sensitivities, Sensitivities2, BumpType, SensitivityMethod
from Quantifi.Toolkit.Curves import CurveUtil

# Import Quantifi Rest API client functions
from qservices.QuantifiApiClient import *

**Set Access Parameters**

Next, we define some access parameters for loading data from Quantifi Risk Database:

In [ ]:
# Define database access parameters and pricing date

# For locally installed Quantifi Risk
os.environ["Q_DATA_SERVICE_URL_PREFIX"] = "https://hb-dc1ut-qfidsp:47200"
os.environ["Q_MARKET_ENV_SERVICE_URL_PREFIX"] = "https://hb-dc1ut-qfidsp:47201"
os.environ["Q_LOCAL_INSTALL"] = "True"
# You are modifying environment variables — but only for the current Python process and its children, not the entire operating system.

# Suppress warnings due to unverified HTTPS certificate
requests.packages.urllib3.disable_warnings()

: 

In [6]:
print(Util.get_data_service_url_prefix())
Util.get_data_service_url_prefix('https://hb-dc1ut-qfidsp:47200')

https://hb-dc1ut-qfidsp:47200


'https://hb-dc1ut-qfidsp:47200'

***

### Load Static Data From the Database

We can load static data from the database, using HQL or functions specific to some types of data. In this example, we are loading a definition of a bond.

In [5]:
# Load a bond product definition from Quantifi Database
bond_info = Connect.get_bond_data(name = 'ATOS 5% Dec30', ds_url="https://hb-dc1ut-qfidsp:47200", verify_https_cert = False)
display(bond_info)

QException: HTTP Response code: 500, with message [{
  "message": "No IPEndpoints were found for host hb-dc1ut-qfiap1.nyc.gerbercapital.com.",
  "code": 500
}] from server.

For the above bond, we can load issuer static data:

In [4]:
# Get Issuer name from the above bond
le_name = bond_info['Issuer']['Name']

# Load Issuer (Legal Entity) definition from the database
le_info = Connect.get_legal_entity_data(name = le_name, verify_https_cert = False)
display(le_info)

{'$type': 'LegalEntity',
 'ObjectId': '28147497671076682',
 'Name': 'Atos SE',
 'LongName': 'Atos SE',
 'Ticker': 'ATOS',
 'CLIP': 'FF4C81',
 'LEI': '5493001EZOOA66PTBR68',
 'Tags': [{'$type': 'Tag', 'Name': 'PrevCLIP', 'Value': 'FF4BG0'}],
 'Country': {'$type': 'Country',
  'ISOAlpha3': 'FRA',
  'ObjectId': '55732045388709960'},
 'CreditGrade': 'None',
 'CapitalStructureRatings': {},
 'EntityRoles': ['Reference'],
 'DefaultRestructuring': 'ModifiedModified',
 'RedValidFrom': '2011-07-01',
 'RatingActions': [],
 'Sectors': [{'$type': 'SectorItem',
   'Sector': {'$type': 'Sector',
    'SectorType': {'$type': 'SectorType',
     'Name': 'Markit Sector',
     'ObjectId': '38562071809359873'},
    'Name': 'Technology',
    'Value': 0.0,
    'ObjectId': '38843546786070572'}}],
 'CurveCurrency': 'None',
 'CurveSeniority': 'SeniorSecured'}

***

### Loading Pricer Objects from the Risk Engine

We can load pricer objects for products and trades defined in the Quantifi Risk application. The pricer objects integrate static data, market data and a pricing model and provide risk measure calculations for all product types in Quantifi.

This functionality relies on a **Calculation Environment** being set up in the Risk Application. Calculation Environment definition controls the settings for the pricing models and risk calculation, as well as market data used for pricing.  

The following cell loads a **BondPricer** object from the Qauntifi Risk application  and some of its properties. Note that the pricer comes with all of the data (static data, curves, market quotes, etc) which are necessary for pricing:

In [6]:
cdx_pricer = Connect.get_pricer_by_trade_id(as_of = '2025-03-04', 
                                             trade_id = "CDX0022186", 
                                             calc_env = 'Official', 
                                             verify_https_cert = False)


dir(cdx_pricer)




['<clr_ctor>',
 'AccrualDays',
 'Accrued',
 'AdjustDurationForRemainingNotional',
 'ApproximateForFastCalculation',
 'AsOf',
 'Basis',
 'BasisFactor',
 'BreakEvenPremium',
 'BuildPaymentPricer',
 'CDX',
 'Carry',
 'Cashflow',
 'Clone',
 'CollectValidationErrors',
 'CurrentNotional',
 'CurrentRate',
 'DiscountCurve',
 'DiscountToSettle',
 'EffectiveNotional',
 'Equals',
 'EquivCDSFwdPremium01',
 'EquivCDSRate01',
 'EquivCDSRecovery01',
 'EquivCDSSpread01',
 'EquivCDSSpreadGamma',
 'EquivCDSTheta',
 'EquivalentCDSPricer',
 'ExpectedLoss',
 'Factor',
 'FullPrice',
 'FwdPremium',
 'FwdPremium01',
 'GenerateCashflow',
 'GeneratePayments',
 'GetDerivativesWrtOrdinates',
 'GetHashCode',
 'GetPaymentSchedule',
 'GetType',
 'HasAnalyticDerivatives',
 'ImpliedQuotedSpread',
 'IntrinsicBreakEvenPremium',
 'IntrinsicRiskyDuration',
 'IntrinsicSurvivalProbability',
 'IntrinsicValue',
 'IntrinsicValue_direct',
 'IsTerminated',
 'MTMCarry',
 'MarkToMarketFactor',
 'MarketPremium',
 'MarketPrice',
 'M

In [16]:


print('              Bond:', cdx_pricer.CDX)
print('             Quote:', cdx_pricer.MarketQuote * 100)
print('    Discount Curve:', cdx_pricer.DiscountCurve.Name)
print('    Survival Curve:', cdx_pricer.SurvivalCurves[99])
print('Scaling:', dir(cdx_pricer.Scaling))
# print()
# print(cdx_pricer.DiscountCurve)
# print(cdx_pricer.SurvivalCurve)

              Bond: CDX-NAHYS43V1-5Y
             Quote: 108.0
    Discount Curve: USDLIBOR
    Survival Curve: CDX.NA.HY.43-V1/SIRIXMAB.USD.SeniorUnsecured.Excluded14[SurvivalFitCalibrator]
 name_ = CDX.NA.HY.43-V1/SIRIXMAB.USD.SeniorUnsecured.Excluded14
 category_ = None
 asOf_ = 04-Mar-2025
 spread_ = 0
 dc_ = Actual/365 Fixed
 freq_ = Continuous
 ccy_ = USD
 interp_ = Quantifi.Toolkit.Numerics.Weighted
 flags_ = 0
 jumpDate_ = <null>
 dt_[0] = 23-Dec-2025 ; y_[0] = 0.99384578
 dt_[1] = 22-Dec-2026 ; y_[1] = 0.97774878
 dt_[2] = 21-Dec-2027 ; y_[2] = 0.94980064
 dt_[3] = 21-Dec-2029 ; y_[3] = 0.854373
 dt_[4] = 23-Dec-2031 ; y_[4] = 0.74834542
 dt_[5] = 21-Dec-2034 ; y_[5] = 0.62681827
tenors_.Count = 6; Tenor list:
tenors_[0]: Name = 1 Years; Product = 1 Years; MarketPv = 0; Quote = 0.0045602368055518, CreditSpread; modelPv_ = 0; weight_ = 1
tenors_[1]: Name = 2 Years; Product = 2 Years; MarketPv = 0; Quote = 0.00736664913689548, CreditSpread; modelPv_ = 0; weight_ = 1
tenors_[2]: 

We can now use the pricer object loaded above to calculate some risk measures for the bond:

In [12]:
def print_pr(pricer):
    #print('              Bond:', pricer.Bond)  
    print('             Quote:', pricer.MarketQuote * 100)
    print('           Accrued: {0:,.2f}'.format(pricer.Accrued()))
    # print('         PV from Q: {0:,.2f}'.format(pricer.ProductPvFromQuote()))  
    # print('               YTM: {0:.2%}'.format(pricer.YieldToMaturity()))
    # print('      Mod Duration: {0:.2f}'.format(pricer.ModDuration()))
    # print('Implied CDS Spread: {0:.2f}'.format(pricer.ImpliedCDSLevel() * 10000))

print_pr(cdx_pricer)

             Quote: 107.38350347970169
           Accrued: -328,125.00


Next, we show that we can manipulate the embedded data (in this case, a Market Quote) to run scenarios on the loaded bond pricer: 

In [13]:
cdx_pricer.QuotingConvention = QuotingConvention.FlatPrice
cdx_pricer.MarketQuote = 1.08 # no method to re-build pricer, although cheeck cds scalinbg factors if they are recomputed or not

print_pr(cdx_pricer)

             Quote: 108.0
           Accrued: -328,125.00


In the following cell, we change change the yield of the bond, as well as use different discount and survival curves for pricing:

In [ ]:
new_discount_curve = Connect.get_discount_curve(as_of = '2022-04-11', curve_name = 'FedFunds', calc_env='Official', verify_https_cert = False)
new_credit_curve = Connect.get_survival_curve(as_of = '2022-04-11', curve_name = 'LTR.USD.SeniorUnsecured.Excluded14', calc_env = 'Official', verify_https_cert = False)

bond_pricer.QuotingConvention = QuotingConvention.Yield
bond_pricer.MarketQuote = 0.05
bond_pricer.DiscountCurve = new_discount_curve
bond_pricer.SurvivalCurve = new_credit_curve

print_pr(bond_pricer)

***

### Load a Portfolio from the Database

Next section loads portfolios of bond trades (or more specifcally *trade pricer* obects) from The Quantifi Risk system, based on a grouping defined by 'Strategy'. Quantifi pricer objects contain the pricing models which compute various risk measures (such as PVs, greeks, etc) for the portfolio of trades.

First, we load the list of trades in the chosen *Strategy*:


In [18]:
# Load portfolio of bond trades for a given strategy. Note that default values of max_results = 500
hql_query = "FROM Trade b WHERE b.Strategy.Name = 'Manual'"
bond_ptf = Connect.data_query(hql_query, max_results = 5000, verify_https_cert = False)
#display(bond_ptf)

# Extract trade IDs, store in a list
trade_ids = [trade['TradeId'] for trade in bond_ptf]
print('Number of trades:', len(trade_ids))
#display(trade_ids)


Number of trades: 4366


Now, we can load the pricers for selected trades (load the first 100):

In [20]:
tic = time.time()

# Load pricers corresponding to the list of trade IDs (they are returned in a dictionary indexed by tradeId). Loading the first 100 pricers.
bond_pricers = Connect.get_pricers_by_trade_ids(as_of='2025-03-04', trade_ids=trade_ids[:100], calc_env='Official', verify_https_cert = False)  # get_pricers_by_trade_ids() important

toc = time.time()
print(round(toc-tic, 2), 'sec total')
print(round((toc-tic)/len(trade_ids), 2), 'sec per trade')
print()

Error(s) occurred when getting pricers. Set with_errors flag to True in method call to get errors object as a sencond return value.
Error(s) occurred when getting pricers. Set with_errors flag to True in method call to get errors object as a sencond return value.
Error(s) occurred when getting pricers. Set with_errors flag to True in method call to get errors object as a sencond return value.
Error(s) occurred when getting pricers. Set with_errors flag to True in method call to get errors object as a sencond return value.
Error(s) occurred when getting pricers. Set with_errors flag to True in method call to get errors object as a sencond return value.
9.73 sec total
0.0 sec per trade



Next, we compute a set of risk measures for each bond and store in a Pandas dataframe:

In [23]:
def create_results_df(bond_pricers):

    Bond_trades_df = pd.DataFrame(index = bond_pricers.keys())  

    # iterate trhough each row and calculate risks for each trade
    for name, pricer in bond_pricers.items():

        pricer.Notional = 10000000

       # Bond_trades_df.at[name, 'Bond Name'] = pricer.Bond
        Bond_trades_df.at[name, 'Notional'] = pricer.Notional
       # Bond_trades_df.at[name, 'Dirty Price'] = pricer.FullPrice() * 100     
        Bond_trades_df.at[name, 'Accrued'] = pricer.Accrued() 
        # Bond_trades_df.at[name, 'Yield'] = pricer.YieldToMaturity()       
        # Bond_trades_df.at[name, 'PV01'] = pricer.PV01() * 10000
        # Bond_trades_df.at[name, 'Yield Convexity'] = pricer.Convexity()
        # Bond_trades_df.at[name, 'Mod Duration'] = pricer.ModDuration()
        Bond_trades_df.at[name, 'Pv'] = pricer.Pv()
        # Bond_trades_df.at[name, 'Z-Spread'] = pricer.ImpliedZSpread() * 10000
        # Bond_trades_df.at[name, 'Implied CDS Level'] = pricer.ImpliedCDSLevel() * 10000

    return Bond_trades_df

def print_results(bond_trade_df):
    # Format and print out results
    format_dict = {}
    for col_name in [ 'Accrued', 'Pv', 'Notional',]: format_dict[col_name] = '{0:,.2f}'
    #for col_name in [ 'Yield' ]: format_dict[col_name] = '{0:.2%}'
    #for col_name in [ 'Quote', 'Dirty Price', 'Mod Duration', 'Z-Spread Duration', 'IR Duration', 'Spread Duration', 'Spread Convexity', 'Yield Convexity', 'PV01', 
    #                  'Implied CDS Level', 'Implied CDS Basis', 'Model Dirty Price', 'Z-Spread', 'ASW Spread', 'Rate Convexity']: format_dict[col_name] = '{0:.2f}'
    format_dict['Maturity'] = '{:%d-%m-%y}'

    df_style = bond_trade_df.style.format(format_dict)

    total_pv = bond_trade_df['Pv'].sum()
    print('Total PV: {0:,.2f}'.format(total_pv))

    display(df_style)

results_df = create_results_df(bond_pricers)
print_results(results_df)

Total PV: -2,194,998.15


,Notional,Accrued,Pv
CDS0019289,"10,000,000.00",0.00,0.00
CDS0019886,"10,000,000.00",0.00,0.00
CDS0021755,"10,000,000.00",0.00,0.00
CDX0013891,"10,000,000.00",0.00,0.00
CDX0014046,"10,000,000.00",0.00,0.00
CDX0014047,"10,000,000.00",0.00,0.00
CDX0014048,"10,000,000.00",0.00,0.00
CDX0014066,"10,000,000.00",0.00,0.00
CDX0014133,"10,000,000.00",0.00,0.00
CDX0014142,"10,000,000.00",0.00,0.00


***
We can compute sensitivities for our portfolio of bonds using Quantifi's sensitivity calculators. As an example, here we compute IR sensitivity for the above portfolio:

In [24]:
# Rates Sensitivty calculator
tic = time.time()

results_table = Sensitivities.Rate(pricers = list(bond_pricers.values()), 
                                   measure = 'Pv', 
                                   initialBump = 0.0, 
                                   upBump = 10.0, 
                                   downBump = 0, 
                                   bumpRelative = False, 
                                   scaledDelta = True,
                                   bumpType = BumpType.ByTenor, 
                                   bumpTenors = None, 
                                   calcGamma = False, 
                                   calcHedge = False, 
                                   hedgeTenor = None, 
                                   recalibrate = False, 
                                   dataTable = None, 
                                   curveFitmethod = None,  
                                   rescaleStrikes = None, 
                                   discountInterp = None)
toc = time.time()

#set up a pandas dataframe to store sensitivities (for ease of use)
df = results_table.to_dataframe()

print('Time, s:', round(toc-tic, 2))

# Print out sensitivities
display(df.pivot_table(index='Curve Tenor', columns='Element', values='Delta', aggfunc=np.sum, margins=True, margins_name='Total').drop(columns = ['Total']).style.format('{:,.0f}'))


Time, s: 1.11


C:\Users\mmoore\AppData\Local\Temp\ipykernel_42596\1850209799.py:29: FutureWarning: The provided callable <function sum at 0x000001B5F9E76020> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  display(df.pivot_table(index='Curve Tenor', columns='Element', values='Delta', aggfunc=np.sum, margins=True, margins_name='Total').drop(columns = ['Total']).style.format('{:,.0f}'))
C:\Users\mmoore\AppData\Local\Temp\ipykernel_42596\1850209799.py:29: FutureWarning: The provided callable <function sum at 0x000001B5F9E76020> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  display(df.pivot_table(index='Curve Tenor', columns='Element', values='Delta', aggfunc=np.sum, margins=True, margins_name='Total').drop(columns = ['Total']).style.format('{:,.0f}'))


Element,EURIBOR
Curve Tenor,
EURIBOR_6 Months.MM_1 Months,5
EURIBOR_6 Months.MM_3 Months,20
EURIBOR_6 Months.MM_6 Months,7
EURIBOR_6 Months.Swap_1 Years,0
EURIBOR_6 Months.Swap_2 Years,0
EURIBOR_6 Months.Swap_3 Years,0
EURIBOR_6 Months.Swap_4 Years,0
EURIBOR_6 Months.Swap_5 Years,0
EURIBOR_6 Months.Swap_6 Years,0


***
### FX Examples

In the following example, we load a set of pricers for FX Forwards and options. We don't need to explicitly set them up, loaded pricers will contain all relevant information and objecyts representing the trades will be of a correct type for each product.

In [ ]:
# Load an FX Forward trade
pricer = Connect.get_pricer_by_trade_id(as_of = '2019-05-09', 
                                        trade_id = 'FxForward0024431', 
                                        calc_env = 'Official', 
                                        verify_https_cert = False)

print('Product:')
print(pricer.FxForward.Description)
print(pricer.FxForward.Maturity)
print(str(pricer.FxForward.PayCcy) + '/' + str(pricer.FxForward.ReceiveCcy))

print('\nPricing:')
print("PV        :", pricer.Pv())
print("SpotFX    :", pricer.SpotFxRate(pricer.FxForward.PayCcy, pricer.FxForward.ReceiveCcy))
print("PayAmount :", pricer.PayAmount())
print("Fwd points:", pricer.ForwardFxPoints(pricer.FxForward.PayCcy, pricer.FxForward.ReceiveCcy))



In [ ]:
# Load an FX option trade
pricer = Connect.get_pricer_by_trade_id(as_of = '2019-05-09', 
                                        trade_id = 'FxOption0023021', 
                                        calc_env = 'Official', 
                                        verify_https_cert = False)
print('Product:')
print(pricer.FxOption)
print(pricer.FxOption.Type)
print(pricer.FxOption.Style)
print(pricer.FxOption.Maturity)

print('\nPricing:')
print("PV    :", pricer.Pv())
print("Delta :", pricer.Delta())
print("Gamma :", pricer.Gamma())
print("Vega  :", pricer.Vega())


In [ ]:
# Load portfolio of FX Forwards for a given strategy. Note that default values of max_results = 500
hql_query = "FROM Trade b WHERE b.TradeId LIKE 'FXForward%'"
options_ptf = Connect.data_query(hql_query, verify_https_cert = False)

# Extract trade IDs, store in a list
trade_ids = [trade['TradeId'] for trade in options_ptf]
print('Number of trades:', len(trade_ids))

pricers = Connect.get_pricers_by_trade_ids(as_of = '2019-05-09', 
                                           trade_ids = trade_ids, 
                                           calc_env = 'Official', 
                                           verify_https_cert = False)

trades_df = pd.DataFrame(index = trade_ids)
for tradeId, pricer in pricers.items():
    trades_df.at[tradeId, 'Product'] = str(pricer.FxForward.PayCcy) + '/' + str(pricer.FxForward.ReceiveCcy)
    trades_df.at[tradeId, 'Maturity'] = pricer.FxForward.Maturity
    trades_df.at[tradeId, 'Fwd Points'] = pricer.ForwardFxPoints(pricer.FxForward.PayCcy, pricer.FxForward.ReceiveCcy)
    trades_df.at[tradeId, 'PV'] = pricer.Pv()

display(trades_df.style.format({'PV':'{:,.2f}', 'Fwd Points':'{:.0f}', 'Maturity':'{:%d-%m-%Y}'}))

***

### Equity Products Examples

In the following example, we load a set of pricers for a single option and a portfolio of options. We don't need to explicitly set them up, loaded pricers will contain all relevant information and objecyts representing the trades will be of a correct type for each product.

In [ ]:
# Load a equity option trade
pricer = Connect.get_pricer_by_trade_id(as_of = '2019-05-09', 
                                        trade_id = 'StockOption24315', 
                                        calc_env = 'Official', 
                                        verify_https_cert = False)
print('Product')
print(pricer.StockOption)
print(pricer.StockOption.Type)
print(pricer.StockOption.Style)
print(pricer.StockOption.Maturity)

print('\nStock Curve')
display(pricer.StockCurve)

print('\nPricing')
print("Vol:    ", pricer.Volatility)
print("PV:     ", pricer.Pv())
print("Delta:  ", pricer.Delta())
print("Gamma:  ", pricer.Gamma())
print("Vega:   ", pricer.Vega())
print("Rho:    ", pricer.Rho())


***

**Option portfolio:**

In [ ]:
# Load portfolio of bond trades for a given strategy. Note that default values of max_results = 500
hql_query = "FROM Trade b WHERE b.TradeId LIKE 'StockOption%'"
options_ptf = Connect.data_query(hql_query, verify_https_cert = False)

# Extract trade IDs, store in a list
trade_ids = [trade['TradeId'] for trade in options_ptf]
print('Number of trades:', len(trade_ids))

pricers = Connect.get_pricers_by_trade_ids(as_of = '2019-05-09', 
                                           trade_ids = trade_ids, 
                                           calc_env = 'Official', 
                                           verify_https_cert = False)

trades_df = pd.DataFrame(index = trade_ids)
for tradeId, pricer in pricers.items():
    trades_df.at[tradeId, 'Product'] = pricer.Product
    trades_df.at[tradeId, 'PV'] = pricer.Pv()
    trades_df.at[tradeId, 'Delta'] = pricer.Delta()
    trades_df.at[tradeId, 'Gamma'] = pricer.Gamma()
    trades_df.at[tradeId, 'Vega'] = pricer.Vega()
    trades_df.at[tradeId, 'Rho'] = pricer.Rho()

display(trades_df.style.format({'PV':'{:,.2f}'}))

In [ ]:
from Quantifi.Toolkit.Base import Dt
from Quantifi.Toolkit.Sensitivity import ThetaFlags, SensitivityRescaleStrikes

thetas = Sensitivities.Theta(pricers = list(pricers.values()), 
                             measure = 'Pv', 
                             toAsOf = Dt(20190510), 
                             toSettle = Dt(20190511), 
                             flags = ThetaFlags.Recalibrate, 
                             rs = [SensitivityRescaleStrikes.UsePricerSetting]*len(pricers))

#set up a pandas dataframe to store sensitivities (for ease of use)
df = pd.DataFrame({'Pricer':pricers.keys(), 'Theta':thetas})

# Print out thetas
display(df.style.format({'Theta':'{:,.2f}'}))


***
### Commodity Examples

In the following examples, we load a set of pricers for Commodity Futures and Listed Future Options. We don't need to explicitly set them up, loaded pricers will contain all relevant information and objecyts representing the trades will be of a correct type for each product.

In [ ]:
# Load a Commodity Forward trade
pricer = Connect.get_pricer_by_trade_id(as_of = '2019-05-09', 
                                        trade_id = 'CommodityFuture0025678', 
                                        calc_env = 'Official', 
                                        verify_https_cert = False)
print("Currency: ", pricer.Product.Ccy)
print("Maturity: ", pricer.Product.EffectiveMaturity)
print("Quote:    ", pricer.QuotedPrice)
print("Notional: ", pricer.Notional)
print("PV:       ", pricer.Pv())

In [ ]:
# Load a Commodity Option trade
pricer = Connect.get_pricer_by_trade_id(as_of = '2019-05-09', 
                                        trade_id = 'CommodFutListOpt0025083', 
                                        calc_env = 'Official', 
                                        verify_https_cert = False)
print('Product:')
print("Option:   ", pricer.CommodityFutureOption)
print("Expiry:   ", pricer.Product.EffectiveMaturity)
print("Notional: ", pricer.Notional)

print('\nCommodity Curve')
display(pricer.CommodityCurve)

print('\nPricing:')
print("PV:    ", pricer.Pv())
print("Delta: ", pricer.Delta())
print("Gamma: ", pricer.Gamma())
print("Vega:  ", pricer.Vega())
print("Charm: ", pricer.Charm())

***

**Listed Commodity Future Option Portfolio:**

In [ ]:
trade_ids = ['CommodFutListOpt0025082', 'CommodFutListOpt0025083', 'CommodFutListOpt0025087', 'CommodFutListOpt0025088']
pricers = Connect.get_pricers_by_trade_ids(as_of = '2019-05-09', 
                                           trade_ids = trade_ids, 
                                           calc_env = 'Official', 
                                           verify_https_cert = False)

trades_df = pd.DataFrame(index = trade_ids)
for tradeId, pricer in pricers.items():
    trades_df.at[tradeId, 'Product'] = pricer.CommodityFutureOption
    trades_df.at[tradeId, 'Expiry'] = pricer.Product.EffectiveMaturity 
    trades_df.at[tradeId, 'PV'] = pricer.Pv()
    trades_df.at[tradeId, 'Delta'] = pricer.Delta()
    trades_df.at[tradeId, 'Gamma'] = pricer.Gamma()
    trades_df.at[tradeId, 'Vega'] = pricer.Vega()
    trades_df.at[tradeId, 'Charm'] = pricer.Charm()

display(trades_df.style.format({'PV':'{:,.2f}', 'Delta':'{:,.2f}', 'Gamma':'{:,.2f}', 'Vega':'{:,.2f}', 'Charm':'{:,.2f}', 'Expiry':'{:%d-%m-%Y}'}))

***

### Load Pricers for Mixed Products

In the following example, we load a set of pricers for a diverse set of products (Bonds, Swaps, CDS, Credit Index). We don't need to explicitly set them up, loaded pricers will contain all relevant information and objecyts representing the trades will be of a correct type for each product.

In [ ]:
trade_ids = ['Bond0024393','Bond0024408','Bond0024406','Bond0024364','Bond0024402',
             'CDX0024485', 'CDX0024471',
             'CDS0024932', 'CDS0024931', 'CDS0024914',
             'Swap0020807', 'Swap0024436', 'Swap0025485',
             'StockOption24311', 'StockOption24316', 'StockOption24319', 'StockOption24320',
             'FxForward0024447', 'FxForward0024423', 'FxOption0023021', 'FxOption0023017']

pricers = Connect.get_pricers_by_trade_ids(as_of = '2019-05-09', 
                                           trade_ids = trade_ids, 
                                           calc_env = 'Official', 
                                           verify_https_cert = False)

trades_df = pd.DataFrame(index = trade_ids)
for tradeId, pricer in pricers.items():
    trades_df.at[tradeId, 'Product'] = pricer.Product
    trades_df.at[tradeId, 'PV'] = pricer.Pv()

display(trades_df.style.format({'PV':'{:,.2f}'}))

We can now calculate aggregated sensitivities for the combined portfolio of trades. Using bucketed spread sensitivity as an example:

In [25]:
# Spread Sensitivity calculator
results_table = Sensitivities.Spread(pricers = list(pricers.values()), 
                                     measure = 'Pv', 
                                     targetQuoteType = QuotingConvention.CreditSpread, 
                                     initialBump = 0.0, 
                                     upBump = 1.0, 
                                     downBump = 0, 
                                     bumpRelative = False, 
                                     scaledDelta = True,
                                     bumpType = BumpType.ByTenor, 
                                     bumpTenors = None, 
                                     calcGamma = True, 
                                     calcHedge = False, 
                                     hedgeTenor = None, 
                                     sensitivityMethod = SensitivityMethod.FiniteDifference,
                                     dataTable = None, 
                                     rescaleStrikes = None)

#set up a pandas dataframe to store sensitivities (for ease of use)
df = results_table.to_dataframe()

# Print out sensitivities
display(df.pivot_table(index='Element', columns='Curve Tenor', values='Delta', aggfunc=np.sum).style.format('{:,.0f}'))

NameError: name 'pricers' is not defined

***
## "Complex" products

Load pricers for 'complex' structured credit products such as CDOs, index tranches and index trades with 100+ underlying credits. 

In [28]:
# Load a set of complex pricers
trade_ids = ['CDXTranche0021876',]

pricers = Connect.get_pricers_by_trade_ids(as_of = '2025-03-04', 
                                           trade_ids = trade_ids, 
                                           calc_env = 'Official', 
                                           verify_https_cert = False)

trades_df = pd.DataFrame(index = trade_ids)
for tradeId, pricer in pricers.items():
    trades_df.at[tradeId, 'Product'] = pricer.Product
    trades_df.at[tradeId, 'PV'] = pricer.Pv()

display(trades_df.style.format({'PV':'{:,.2f}'}))

print("\nCDO CDO0024669 Survival Curves:")
display(list(pricers["CDXTranche0021876"].SurvivalCurves))

,Product,PV
CDXTranche0021876,CDXTranche0021876,"2,161,621.73"



CDO CDO0024669 Survival Curves:


[<CLR: iTraxx Europe Series 36 Version 1/EDF.EUR.SeniorUnsecured.ModifiedModified14[SurvivalFitCalibrator]
  name_ = iTraxx Europe Series 36 Version 1/EDF.EUR.SeniorUnsecured.ModifiedModified14
  category_ = None
  asOf_ = 04-Mar-2025
  spread_ = 0
  dc_ = Actual/365 Fixed
  freq_ = Continuous
  ccy_ = EUR
  interp_ = Quantifi.Toolkit.Numerics.Weighted
  flags_ = 0
  jumpDate_ = <null>
  dt_[0] = 23-Dec-2025 ; y_[0] = 0.99682139
  dt_[1] = 22-Dec-2026 ; y_[1] = 0.99102657
  dt_[2] = 21-Dec-2027 ; y_[2] = 0.98302671
  dt_[3] = 21-Dec-2029 ; y_[3] = 0.95153914
  dt_[4] = 23-Dec-2031 ; y_[4] = 0.90738373
  dt_[5] = 21-Dec-2034 ; y_[5] = 0.81324036
 tenors_.Count = 6; Tenor list:
 tenors_[0]: Name = 1 Years; Product = 1 Years; MarketPv = 0; Quote = 0.00234568959438184, CreditSpread; modelPv_ = 0; weight_ = 1
 tenors_[1]: Name = 2 Years; Product = 2 Years; MarketPv = 0; Quote = 0.00296064500287428, CreditSpread; modelPv_ = 0; weight_ = 1
 tenors_[2]: Name = 3 Years; Product = 3 Years; Marke